# Auditoría segura de correcciones — urban-v1

Verifica sin entrenamiento: (1) AdamW para semántica y SGD para instancias; (2) ReduceLROnPlateau con `mode=max`, `patience=2`; (3) letterbox con aspecto preservado para imágenes/máscaras/cajas; (4) hashes y versiones para reproducibilidad.

**No ejecuta `backward()`, `optimizer.step()`, épocas ni validación experimental. No descarga datasets, no abre ni modifica `test`, no crea checkpoints y mantiene `RUN_TRAINING=False`.** Solo clona/actualiza el código del repositorio y lee los manifiestos/índices TRAIN/VAL ya presentes en Drive.

## 1. Montar Drive y sincronizar el repositorio privado

Usa el secreto `GITHUB_TOKEN` con permiso Contents: Read-only. No pegues tokens en el notebook ni en el chat.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess, getpass, json, hashlib, datetime, platform, sys
drive.mount('/content/drive')
REPO='Leohf04/Procesamiento-de-Im-genes-'; BRANCH='urban-models-and-benchmark'
WORKDIR=Path('/content/Procesamiento-de-Im-genes-')
try: TOKEN=userdata.get('GITHUB_TOKEN')
except Exception: TOKEN=None
if not TOKEN: TOKEN=getpass.getpass('GitHub PAT (Contents: Read-only): ')
if not TOKEN: raise RuntimeError('Falta GITHUB_TOKEN; no se puede leer el repositorio privado.')
askpass=Path('/tmp/urban_git_askpass.sh')
askpass.write_text('#!/bin/sh\\ncase "$1" in *Username*) echo x-access-token ;; *Password*) echo "$COLAB_GITHUB_TOKEN" ;; *) exit 1 ;; esac\\n')
askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':TOKEN})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,f'https://github.com/{REPO}.git',str(WORKDIR)],check=True,env=env)
del TOKEN, env
os.chdir(WORKDIR); sys.path.insert(0,str(WORKDIR))
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('HEAD:',HEAD); print('RUN_TRAINING=False')
DRIVE_ROOT=Path('/content/drive/MyDrive/ProcesamientoImagenes')
DATA_DIR=DRIVE_ROOT/'urban_data_capacity'; INDEX_DIR=DATA_DIR/'local_annotation_index'
OUT_DIR=DRIVE_ROOT/'auditoria_correcciones_segura'; OUT_DIR.mkdir(parents=True,exist_ok=True)


## 2. Pruebas sintéticas de optimizador, scheduler y geometría

Se usa un modelo lineal diminuto y arrays sintéticos. No se pasa ningún batch del dataset ni se actualizan parámetros.

In [ ]:
import torch, numpy as np
from PIL import Image
from src.urban_optimizer import build_optimizer_and_scheduler
from src.urban_geometry import letterbox_geometry, letterbox_image, letterbox_mask
checks={}
toy=torch.nn.Linear(3,2)
opt_sem,sch_sem=build_optimizer_and_scheduler(toy,'semantic',{'learning_rate':1e-3})
checks['semantic_optimizer']=type(opt_sem).__name__=='AdamW'
checks['semantic_scheduler']=type(sch_sem).__name__=='ReduceLROnPlateau' and sch_sem.mode=='max' and sch_sem.patience==2
opt_ins,sch_ins=build_optimizer_and_scheduler(toy,'instances',{'learning_rate':1e-3})
checks['instances_optimizer']=type(opt_ins).__name__=='SGD'
checks['instances_scheduler']=type(sch_ins).__name__=='ReduceLROnPlateau' and sch_ins.mode=='max' and sch_ins.patience==2
g=letterbox_geometry((100,200),(512,512))
checks['letterbox_geometry']=(g.resized_height,g.resized_width,g.top,g.left)==(256,512,128,0)
im=Image.fromarray(np.full((100,200,3),127,dtype=np.uint8),'RGB')
im_out,ig=letterbox_image(im,(512,512))
checks['image_shape_and_padding']=im_out.size==(512,512) and np.asarray(im_out)[0,0].tolist()==[0,0,0] and np.asarray(im_out)[128,20].tolist()==[127,127,127]
mask=np.full((100,200),7,dtype=np.uint8)
mask_out,mg=letterbox_mask(mask,(512,512),fill_value=255)
checks['semantic_mask_padding']=mask_out.shape==(512,512) and int(mask_out[0,0])==255 and int(mask_out[128,20])==7
inst=np.zeros((100,200),dtype=np.uint8); inst[20:60,40:100]=3
inst_out,ig2=letterbox_mask(inst,(512,512),fill_value=0)
ys,xs=np.where(inst_out==3)
box=ig2.boxes_xyxy(np.array([[40,20,100,60]],dtype=np.float32))
checks['instance_mask_and_box_geometry']=len(xs)>0 and np.allclose(box[0],[102.4,179.2,256.0,281.6],atol=1.1)
checks['no_optimizer_step']=all(p.grad is None for p in toy.parameters())
print(json.dumps(checks,indent=2)); print('PASS:',sum(checks.values()),'/',len(checks))
if not all(checks.values()): raise AssertionError('Una o más pruebas sintéticas fallaron.')


## 3. Auditoría de reproducibilidad y soporte TRAIN/VAL

Calcula SHA-256 de manifiestos e índices locales existentes, ontología y fuentes críticas. No inspecciona `test`. Si faltan archivos, el reporte queda `INCOMPLETE`; no se inventan hashes ni soporte.

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
source_paths=['configs/urban_ontology_v1.json','src/urban_geometry.py','src/urban_optimizer.py','src/data_adapters.py','scripts/urban_experiment_factory.py','docs/PROTOCOLO_ENTRENAMIENTO_EXPERIMENTAL.md']
hashes={}
for rel in source_paths:
    p=WORKDIR/rel
    if p.is_file(): hashes[rel]=sha256_file(p)
data_files=[]; missing=[]
for ds in ['a2d2','urbansyn']:
    for split in ['train','val']:
        candidates=[DATA_DIR/'local_manifests'/f'{ds}_{split}.jsonl', INDEX_DIR/f'{ds}_{split}.annotations.jsonl']
        for p in candidates:
            if p.is_file(): data_files.append(p)
            else: missing.append(str(p))
hashes.update({str(p.relative_to(DATA_DIR)):sha256_file(p) for p in data_files})
support={}; row_counts={}
for ds in ['a2d2','urbansyn']:
  for split in ['train','val']:
    p=INDEX_DIR/f'{ds}_{split}.annotations.jsonl'
    if not p.is_file(): continue
    rows=[json.loads(x) for x in p.read_text(encoding='utf-8').splitlines() if x.strip()]
    row_counts[f'{ds}_{split}']=len(rows)
    sem={}; inst={}
    for row in rows:
      for k,v in (row.get('semantic_pixels') or {}).items(): sem[k]=sem.get(k,0)+int(v)
      for k,v in (row.get('instance_counts') or row.get('instances_by_class') or {}).items(): inst[k]=inst.get(k,0)+int(v)
    support[f'{ds}_{split}']={'semantic_pixels':sem,'instance_counts':inst,'annotation_status_not_ok':sum(r.get('annotation_status','ok')!='ok' for r in rows)}
import torchvision
audit={
 'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'git_head':HEAD,'python':platform.python_version(),'torch':str(torch.__version__),
 'torchvision':str(torchvision.__version__),'cuda_runtime':torch.version.cuda,
 'cuda_available':torch.cuda.is_available(),'gpu_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
 'checks':checks,'hashes_sha256':hashes,'row_counts_train_val_only':row_counts,
 'class_support_train_val_only':support,'missing_expected_files':missing,
 'test_split_touched':False,'dataset_downloaded':False,'optimizer_steps':0,
 'backward_calls':0,'epochs_started':0,'checkpoint_written':False,'RUN_TRAINING':False,
 'weight_initialization_policy':'No experiment model instantiated in this audit; model weight provenance remains to be frozen by the run config.',\n 'effective_experiment_config_frozen':False,\n 'model_weight_provenance_frozen':False,\n 'experiment_readiness':'BLOCKED_PENDING_CONFIG_AND_WEIGHT_PROVENANCE'
}
audit['status']='PASS' if all(checks.values()) and not missing else 'INCOMPLETE'
report_path=OUT_DIR/'urban_corrections_audit.json'
report_path.write_text(json.dumps(audit,indent=2,ensure_ascii=False),encoding='utf-8')
print('Reporte:',report_path); print('Estado:',audit['status'])
print('Archivos de datos con hash:',len(data_files),'| faltantes:',len(missing))


## 4. Bloque único de continuidad

Copia todo el bloque delimitado. Incluye resultados, hashes, versión y cualquier archivo faltante para continuar en otra conversación.

In [ ]:
payload={'status':audit['status'],'git_head':HEAD,'report_path':str(report_path),'checks':checks,'hashes_sha256':hashes,'row_counts_train_val_only':row_counts,'class_support_train_val_only':support,'missing_expected_files':missing,'optimizer_steps':0,'test_split_touched':False,'RUN_TRAINING':False,'experiment_readiness':audit['experiment_readiness'],'effective_experiment_config_frozen':False,'model_weight_provenance_frozen':False}
block='PROYECTO URBAN-V1 — RESULTADO AUDITORÍA SEGURA\n'+json.dumps(payload,ensure_ascii=False,sort_keys=True)
print('===== BLOQUE_CONTINUIDAD_INICIO =====')
print(block)
print('===== BLOQUE_CONTINUIDAD_FIN =====')
print('JSON persistido en Drive:',report_path)
